# SPAM MAIL DETECTION

In [552]:
import pandas as pd
import re
import nltk
import pickle as pk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

In [553]:
mail_data=pd.read_csv('spam.csv',encoding='latin-1')

In [554]:
mail_data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [555]:
mail_data.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

In [556]:
mail_data.shape

(5572, 5)

In [557]:
for column in mail_data.columns:
    print(f"Column: {column}")
    print(len(mail_data[column].unique()))


Column: v1
2
Column: v2
5169
Column: Unnamed: 2
44
Column: Unnamed: 3
11
Column: Unnamed: 4
6


In [558]:
mail_data.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1, inplace=True)

mail_data.rename(columns={'v1': 'label', 'v2': 'message'}, inplace=True)

In [559]:
mail_data.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Label Encoding

In [560]:
mail_data['label'] = mail_data['label'].map({'spam': 1, 'ham': 0})

In [561]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [562]:
ps = PorterStemmer()

In [563]:
stop_words = set(stopwords.words('english'))

In [564]:
print(stopwords.words('english')[:10])

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']


In [565]:
def preprocess_text(text):
    text = text.lower()  
    text = re.sub(r'http\S+', ' ', text)   
    text = re.sub(r'[^a-zA-Z]', ' ', text)   
    words = word_tokenize(text)
    words = [word for word in words if word not in stop_words]    
    words = [ps.stem(word) for word in words]    
    return " ".join(words)

In [566]:
mail_data['processed_text'] = mail_data['message'].apply(preprocess_text)

spam - 1<br>
ham - 0

In [567]:
X = mail_data['processed_text']
Y = mail_data['label'] 

In [568]:
print(X)

0       go jurong point crazi avail bugi n great world...
1                                   ok lar joke wif u oni
2       free entri wkli comp win fa cup final tkt st m...
3                     u dun say earli hor u c alreadi say
4                    nah think goe usf live around though
                              ...                        
5567    nd time tri contact u u pound prize claim easi...
5568                                b go esplanad fr home
5569                                    piti mood suggest
5570    guy bitch act like interest buy someth els nex...
5571                                       rofl true name
Name: processed_text, Length: 5572, dtype: object


In [569]:
print(Y)

0       0
1       0
2       1
3       0
4       0
       ..
5567    1
5568    0
5569    0
5570    0
5571    0
Name: label, Length: 5572, dtype: int64


In [570]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=3)

In [571]:
print(X.shape)
print(X_train.shape)
print(X_test.shape)

(5572,)
(4457,)
(1115,)


In [572]:
vectorizer = TfidfVectorizer()
X_train_features = vectorizer.fit_transform(X_train)
X_test_features = vectorizer.transform(X_test)


In [573]:
print(X_train)

3075    mum hope great day hope text meet well full li...
1787                                   ye sura sun tv lol
1614                     sef dey laugh meanwhil darl anji
4304                                   yo come carlo soon
3266                               ok come n pick u engin
                              ...                        
789                            gud mrng dear hav nice day
968                                 will go aptitud class
1667          dad gon na call get work ask crazi question
3321    ok darlin supos ok worri much film stuff mate ...
1688                           nan sonathaya soladha boss
Name: processed_text, Length: 4457, dtype: object


In [574]:
print(X_train_features)

  (0, 3123)	0.29473823593356974
  (0, 2196)	0.43070194535584805
  (0, 1991)	0.44582374908675926
  (0, 1140)	0.3688325805843802
  (0, 4832)	0.19671033581522265
  (0, 2933)	0.21758099038206752
  (0, 5343)	0.22139463068022516
  (0, 1825)	0.30299014669442126
  (0, 2695)	0.24418433444341897
  (0, 11)	0.32611021837325754
  (1, 5541)	0.35056971070320353
  (1, 4699)	0.5652509076654626
  (1, 4676)	0.4769136859540388
  (1, 5059)	0.4306015894277422
  (1, 2750)	0.380431198316959
  (2, 4194)	0.45329962489137066
  (2, 1221)	0.3811461667946165
  (2, 2643)	0.34506943774623944
  (2, 2925)	0.41722289584299355
  (2, 1130)	0.3880961710740478
  (2, 186)	0.45329962489137066
  (3, 5564)	0.524841815062256
  (3, 922)	0.34970933876863236
  (3, 730)	0.5979251862237673
  (3, 4464)	0.49470184881344015
  :	:
  (4453, 241)	0.6529435350028746
  (4454, 3149)	0.31741325919431035
  (4454, 1894)	0.2286714339699626
  (4454, 3847)	0.3828677335168324
  (4454, 5457)	0.30522226483287346
  (4454, 691)	0.20579130454519803
  (44

In [575]:
Y_train = Y_train.astype('int')
Y_test = Y_test.astype('int')

In [576]:
smote = SMOTE(random_state=42)

In [577]:
X_train_smote, Y_train_smote = smote.fit_resample(X_train_features, Y_train)

In [578]:
print(Y_train_smote.value_counts())

label
0    3865
1    3865
Name: count, dtype: int64


In [579]:
models = {
    "Logistic Regression": LogisticRegression(),
    "SVM": SVC(kernel='linear', probability=True),
}

In [580]:
for name, model in models.items():

    print("-"*60)
    print(name)
    print("-"*60)

    model.fit(X_train_smote, Y_train_smote)

    train_pred = model.predict(X_train_features)

    print("\nTRAINING RESULTS")
    print("Accuracy:", accuracy_score(Y_train, train_pred))
    print("Confusion Matrix:\n", confusion_matrix(Y_train, train_pred))
    print("Classification Report:\n", classification_report(Y_train, train_pred))

    test_pred = model.predict(X_test_features)

    print("\nTEST RESULTS")
    print("Accuracy:", accuracy_score(Y_test, test_pred))
    print("Confusion Matrix:\n", confusion_matrix(Y_test, test_pred))
    print("Classification Report:\n", classification_report(Y_test, test_pred))

    print("\n\n")

------------------------------------------------------------
Logistic Regression
------------------------------------------------------------

TRAINING RESULTS
Accuracy: 0.9885573255553063
Confusion Matrix:
 [[3838   27]
 [  24  568]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      3865
           1       0.95      0.96      0.96       592

    accuracy                           0.99      4457
   macro avg       0.97      0.98      0.98      4457
weighted avg       0.99      0.99      0.99      4457


TEST RESULTS
Accuracy: 0.97847533632287
Confusion Matrix:
 [[951   9]
 [ 15 140]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.99       960
           1       0.94      0.90      0.92       155

    accuracy                           0.98      1115
   macro avg       0.96      0.95      0.95      1115
weighted avg       0.98      0.98  

In [581]:
for name, model in models.items():

    y_prob = model.predict_proba(X_test_features)[:,1]

    auc_score = roc_auc_score(Y_test, y_prob)

    print(f"{name} ROC-AUC Score: {auc_score:.4f}")

Logistic Regression ROC-AUC Score: 0.9953
SVM ROC-AUC Score: 0.9963


In [582]:
with open("spam_models.pkl", "wb") as file:
    pk.dump({
        "logistic_model": models["Logistic Regression"],
        "svm_model": models["SVM"],
        "vectorizer": vectorizer
    }, file)

In [583]:
with open("spam_models.pkl", "rb") as file:
    data = pk.load(file)

logistic_model = data["logistic_model"]
svm_model = data["svm_model"]
vectorizer = data["vectorizer"]

In [584]:
input_mail = ["Congratulations! You have won a FREE lottery worth $10000. Claim your prize now by clicking this link."]

processed_mail = preprocess_text(input_mail[0])
input_features = vectorizer.transform([processed_mail])

lr_pred = logistic_model.predict(input_features)
svm_pred = svm_model.predict(input_features)

print("Logistic Regression:", "Spam" if lr_pred[0] == 1 else "Ham")
print("SVM:", "Spam" if svm_pred[0] == 1 else "Ham")

Logistic Regression: Spam
SVM: Spam


In [585]:
input_mail = ["Hi Shivangi, the meeting has been scheduled for tomorrow at 10 AM. Please join on time."]

processed_mail = preprocess_text(input_mail[0])
input_features = vectorizer.transform([processed_mail])

lr_pred = logistic_model.predict(input_features)
svm_pred = svm_model.predict(input_features)

print("Logistic Regression:", "Spam" if lr_pred[0] == 1 else "Ham")
print("SVM:", "Spam" if svm_pred[0] == 1 else "Ham")

Logistic Regression: Ham
SVM: Ham
